In [1]:
from pyspark.sql import SparkSession

In [2]:
#Must set this env variable to avoid warnigns
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = '1'

In [3]:
import pyspark.pandas as ps

In [4]:
# Initialize Spark Session
spark = SparkSession.builder \
.appName("Pandas API on Spark") \
.config("spark.sql.ansi.enabled", "false") \
.config("spark.executorEnv.PYARROW_IGNORE_TIMEZONE", "1") \
.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-pattern-layout-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/06 14:11:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
#1. Create a pandas-on-Spark DataFrame

ps_df = ps.DataFrame({
    "id": [1,2,3,4,5],
    "name": ["Alice", "Bob", "Charlie", "David", "Emma"],
    "age": [25,30,35,40,45],
    "salary": [50000,60000,75000,80000,120000]
})

In [7]:
print("Pandas-on-Spark DataFrame:")
print(ps_df)

Pandas-on-Spark DataFrame:


   id     name  age  salary
0   1    Alice   25   50000
1   2      Bob   30   60000
2   3  Charlie   35   75000
3   4    David   40   80000
4   5     Emma   45  120000


In [8]:
#2. Perform Pandas-stype operations on Spark
print("\nAverage Age:", ps_df["age"].mean())


Average Age: 35.0


In [11]:
# Compute summary statistics
print("\Summary Statistics:")
print(ps_df.describe())

\Summary Statistics:
             id        age        salary
count  5.000000   5.000000       5.00000
mean   3.000000  35.000000   77000.00000
std    1.581139   7.905694   26832.81573
min    1.000000  25.000000   50000.00000
25%    2.000000  30.000000   60000.00000
50%    3.000000  35.000000   75000.00000
75%    4.000000  40.000000   80000.00000
max    5.000000  45.000000  120000.00000


In [14]:
#3. Apply a function : Add a new column with salary increment
ps_df["salary_after_increment"] = ps_df["salary"] * 1.1
print("\nDataFrame after Salary Increment:")
print(ps_df)


DataFrame after Salary Increment:
   id     name  age  salary  salary_after_increment
0   1    Alice   25   50000                 55000.0
1   2      Bob   30   60000                 66000.0
2   3  Charlie   35   75000                 82500.0
3   4    David   40   80000                 88000.0
4   5     Emma   45  120000                132000.0


In [15]:
#4. Filtering (similar to pandas)
filtered_ps_df = ps_df[ps_df["age"] > 30]
print("\n Filtered DataFrame (age > 30):")
print(filtered_ps_df)


 Filtered DataFrame (age > 30):
   id     name  age  salary  salary_after_increment
2   3  Charlie   35   75000                 82500.0
3   4    David   40   80000                 88000.0
4   5     Emma   45  120000                132000.0


In [16]:
# 5. Convert Pandas-on-spark dataframe to spark dataframe
spark_df = ps_df.to_spark()
print("\n Converted Spark Dataframe")
spark_df.show()

/opt/spark/python/pyspark/pandas/utils.py:1017: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)



 Converted Spark Dataframe
+---+-------+---+------+----------------------+
| id|   name|age|salary|salary_after_increment|
+---+-------+---+------+----------------------+
|  1|  Alice| 25| 50000|     55000.00000000001|
|  2|    Bob| 30| 60000|               66000.0|
|  3|Charlie| 35| 75000|               82500.0|
|  4|  David| 40| 80000|               88000.0|
|  5|   Emma| 45|120000|              132000.0|
+---+-------+---+------+----------------------+



In [17]:
#6. Convert Spark DataFrame back to Pandas-on-Spark DataFrame
ps_df_from_spark = ps.DataFrame(spark_df)
print(ps_df_from_spark)

   id     name  age  salary  salary_after_increment
0   1    Alice   25   50000                 55000.0
1   2      Bob   30   60000                 66000.0
2   3  Charlie   35   75000                 82500.0
3   4    David   40   80000                 88000.0
4   5     Emma   45  120000                132000.0


In [18]:
spark.stop()